In [4]:
import cv2
import time
from ultralytics import YOLO
from pathlib import Path

# --- CONFIGURATION ---
# 1. Path to your best trained YOLOv8 model
MODEL_PATH = Path("../runs/detect/ppe_final_tuned_model2/weights/best.pt")

# 2. Video source: 0 for webcam, or path to a video file
VIDEO_SOURCE = Path("../dataset/source_files/hardhat.mp4")

# 3. Confidence threshold for detections
CONFIDENCE_THRESHOLD = 0.5

# 4. Classes that constitute a violation
#    These must match the names in your data.yaml file exactly.
VIOLATION_CLASSES = {'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest'}

# 5. Alerting configuration
ALERT_COOLDOWN_SECONDS = 5  # Wait 5 seconds before sending another alert


def main():
    """
    Main function to run the real-time PPE detection and alert system.
    """
    # Load the trained YOLOv8 model
    if not MODEL_PATH.exists():
        print(f"Error: Model file not found at '{MODEL_PATH}'")
        return
    model = YOLO(MODEL_PATH)

    # Initialize video capture
    try:
        # Convert Path object to string for VideoCapture
        video_source_str = str(VIDEO_SOURCE) if VIDEO_SOURCE != 0 else 0
        cap = cv2.VideoCapture(video_source_str)
        if not cap.isOpened():
            raise IOError(f"Cannot open video source: {VIDEO_SOURCE}")
    except Exception as e:
        print(f"Error initializing video capture: {e}")
        return

    last_alert_time = 0

    print("Starting real-time detection... Press 'q' to quit.")

    while True:
        success, frame = cap.read()
        if not success:
            if VIDEO_SOURCE != 0:
                print("End of video file reached.")
            else:
                print("Failed to capture frame from webcam.")
            break

        # Run YOLOv8 inference on the frame
        # stream=True is more memory-efficient for video processing
        results = model(frame, stream=True, verbose=False)

        # Store the specific violations detected in this frame
        violations_in_frame = set()
        
        for r in results:
            boxes = r.boxes
            for box in boxes:
                # --- Get detection data ---
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                confidence = box.conf[0]
                class_id = int(box.cls[0])
                class_name = model.names[class_id]

                # --- Filter weak detections ---
                if confidence < CONFIDENCE_THRESHOLD:
                    continue

                # --- Rule Engine: Check for violations ---
                is_violation = class_name in VIOLATION_CLASSES
                if is_violation:
                    violations_in_frame.add(class_name)

                # --- Detection Overlay ---
                # Set color based on whether it's a violation
                color = (0, 0, 255) if is_violation else (0, 255, 0) # Red for violation, Green for safe
                label = f"{class_name}: {confidence:.2f}"

                # Draw bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                # Draw label background
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # --- Alerting Mechanism ---
        current_time = time.time()
        if violations_in_frame and (current_time - last_alert_time > ALERT_COOLDOWN_SECONDS):
            # Create a more specific alert message
            violation_str = ", ".join(sorted(list(violations_in_frame)))
            print(f"ALERT: Safety violation(s) detected: [{violation_str}] at {time.ctime()}!")
            last_alert_time = current_time
            # Here you can integrate other alert systems (e.g., send_email_alert())
            
            # Visual alert on the screen
            cv2.putText(frame, "!!! VIOLATION DETECTED !!!", (50, 50),
                        cv2.FONT_HERSHEY_TRIPLEX, 1.0, (0, 0, 255), 2)


        # Display the output frame
        cv2.imshow("Real-Time PPE Detection", frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # Release resources
    cap.release()
    cv2.destroyAllWindows()
    print("Detection stopped.")

if __name__ == "__main__":
    main()



Starting real-time detection... Press 'q' to quit.
ALERT: Safety violation(s) detected: [NO-Hardhat] at Thu Sep 25 15:41:01 2025!
ALERT: Safety violation(s) detected: [NO-Hardhat, NO-Mask] at Thu Sep 25 15:41:06 2025!
ALERT: Safety violation(s) detected: [NO-Safety Vest] at Thu Sep 25 15:41:12 2025!
ALERT: Safety violation(s) detected: [NO-Safety Vest] at Thu Sep 25 15:41:18 2025!
ALERT: Safety violation(s) detected: [NO-Safety Vest] at Thu Sep 25 15:41:25 2025!
ALERT: Safety violation(s) detected: [NO-Safety Vest] at Thu Sep 25 15:41:30 2025!
ALERT: Safety violation(s) detected: [NO-Safety Vest] at Thu Sep 25 15:41:35 2025!
ALERT: Safety violation(s) detected: [NO-Safety Vest] at Thu Sep 25 15:41:40 2025!
End of video file reached.
Detection stopped.
